<a href="https://colab.research.google.com/github/HakumenWorld/Digital-Skola/blob/main/Febi_Kristiana_HOMEWORK_Exploring_DataFrame_DS_Batch60.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data Diri

Nama          : Febi Kristiana
Digital Skola : Batch 60

In [10]:
#Inserting Data

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

file_path = "/content/drive/MyDrive/Digital Skola/Homework-DataFrame-Dataset.xlsx"

df_paid = pd.read_excel(file_path, sheet_name="Paid-Transaction")
df_seller = pd.read_excel(file_path, sheet_name="Seller")

df_paid.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Paid Date,Order Number,First Name,Last Name,Meta Category,Product Name,Transaction Amount,Seller Discount,Sales Discount,Delivery Fee,Other Discount
0,20170724,201707240088517,elvride,aries,Babies/ Kids,Pineapple Hat Anak Ala Korea - 6M - 4Y - Unise...,300000,153000,10200.0,9000,0.0
1,20170701,201707018889790,BASIR,Ninuk,Service/ Mokado,Pulsa BOLT 150.000,287800,68000,0.0,0,0.0
2,20170707,201707079264675,Citra,Ardi,Service/ Mokado,"XTRA Combo 12X 6GB, 12bln",35000,0,700.0,0,0.0
3,20170720,201707209945714,Dian,Renaldi,Fashion,Square Foldable Travel Bag / Tas Koper Luggage...,85000,0,5900.0,0,0.0
4,20170722,201707220002354,rizal,Tamba,Gadget/ Komputer,Samsung Galaxy Note 5 Gold,81000,0,5600.0,0,0.0


## Soal 1

1.	Buatlah kolom baru bernama “Full Name” dan sisipkan kolom tersebut setelah kolom “Last Name” yang berisikan nama lengkap dari setiap isian record dengan ketentuan sebagai berikut:
•	Nama lengkap menggabungkan “First Name” dan “Last Name”.
•	Nama lengkap menggunakan “title case”, yaitu untuk setiap kata, huruf awalnya harus huruf kapital.


In [11]:
df_paid.insert(
    df_paid.columns.get_loc("Last Name") + 1,
    "Full Name",
    (df_paid["First Name"].astype(str) + " " + df_paid["Last Name"].astype(str)).str.title()
)

df_paid[["First Name", "Last Name", "Full Name"]].head()

,First Name,Last Name,Full Name
0,elvride,aries,Elvride Aries
1,BASIR,Ninuk,Basir Ninuk
2,Citra,Ardi,Citra Ardi
3,Dian,Renaldi,Dian Renaldi
4,rizal,Tamba,Rizal Tamba


## SOAL 2

2.	Tambahkan sebuah kolom setelah kolom “Seller Discount” dan beri nama kolom tersebut sebagai “GMV”. GMV adalah Gross Merchandise Value yang nilainya dihitung berdasarkan “Transaction Amount” dikurangi “Seller Discount” ditambah dengan “Delivery Fee”.

In [12]:
df_paid.insert(
    df_paid.columns.get_loc("Seller Discount") + 1,
    "GMV",
    df_paid["Transaction Amount"] - df_paid["Seller Discount"] + df_paid["Delivery Fee"]
)

df_paid[["Transaction Amount", "Seller Discount", "Delivery Fee", "GMV"]].head()

,Transaction Amount,Seller Discount,Delivery Fee,GMV
0,300000,153000,9000,156000
1,287800,68000,0,219800
2,35000,0,0,35000
3,85000,0,0,85000
4,81000,0,0,81000


## SOAL 3

3.	Kelompokkan setiap Meta Category menjadi 2 group sesuai tabel di bawah ini (hint: buat kolom tambahan yang berisi keterangan group 1 atau 2 sesuai Meta Category-nya). Buatlah DataFrame baru yang menyimpan pivot table dari DataFrame “Paid-Transaction” di atas untuk menampilkan jumlah total nilai GMV per-bulan untuk setiap group.

Group 1

Home/ Food

Sports/ Hobi/ Otomotif

Fashion

Beauty/ Health

|

Group 2

Gadget/ Komputer

Elektronik

Service/ Mokado

Babies/ Kids

In [13]:
df_paid["Paid Date"] = pd.to_datetime(df_paid["Paid Date"].astype(str), format="%Y%m%d")

category_group = {
    "Home/ Food": "Group 1",
    "Sports/ Hobi/ Otomotif": "Group 1",
    "Fashion": "Group 1",
    "Beauty/ Health": "Group 1",
    "Gadget/ Komputer": "Group 2",
    "Elektronik": "Group 2",
    "Service/ Mokado": "Group 2",
    "Babies/ Kids": "Group 2"
}

df_paid["Group"] = df_paid["Meta Category"].map(category_group)
df_paid["Month"] = df_paid["Paid Date"].dt.to_period("M")

pivot_gmv = df_paid.pivot_table(
    index="Month",
    columns="Group",
    values="GMV",
    aggfunc="sum",
    fill_value=0
)

pivot_gmv

Group,Group 1,Group 2
Month,,
2017-07,147724300,230414100
2017-08,94599100,348790700
2017-09,263786400,191482700
2017-10,114883100,150263700
2017-11,263329200,326532200
2017-12,131366800,451421300


### SOAL 4

4.	Buatlah python statement untuk mencari Seller mana yang memiliki total nilai GMV paling tinggi di bulan Agustus 2017. (Hint: gunakan juga sheet “Seller”)

In [14]:
df_merge = df_paid.merge(df_seller, on="Order Number", how="left")

top_seller_august = (
    df_merge[
        (df_merge["Paid Date"].dt.year == 2017) &
        (df_merge["Paid Date"].dt.month == 8)
    ]
    .groupby("Seller", as_index=False)["GMV"]
    .sum()
    .sort_values("GMV", ascending=False)
    .head(1)
)

top_seller_august

,Seller,GMV
207,MOBILEPULSA APP,127241500


## SOAL 5

5.	Buatlah python statement untuk mencari Seller mana yang memiliki banyaknya transaksi (count) paling banyak di bulan September 2017 khusus untuk Meta Category “Fashion”. (Hint: gunakan juga sheet “Seller”)

In [15]:
top_seller_sep_fashion = (
    df_merge[
        (df_merge["Paid Date"].dt.year == 2017) &
        (df_merge["Paid Date"].dt.month == 9) &
        (df_merge["Meta Category"] == "Fashion")
    ]
    .groupby("Seller", as_index=False)["Order Number"]
    .count()
    .rename(columns={"Order Number": "Transaction Count"})
    .sort_values("Transaction Count", ascending=False)
    .head(1)
)

top_seller_sep_fashion

,Seller,Transaction Count
173,tokoaqila,31
